# 02 — Modeling (iterasi 3)
Pipeline scikit-learn:

```
Pipeline
├── struct : StructuralFeatures          kode wilayah, rumah tangga, proksi fisika
├── btype  : BuildingTypeClusterer       (opsional) KMeans tipe bangunan
├── vtype  : VulnerabilityTyper          (opsional) tipe kerentanan supervised
├── prep   : ColumnTransformer
│   ├── num : passthrough
│   ├── cat : CategoricalCoder
│   ├── grp : GroupRelativeFeatures       bangunan relatif terhadap areanya
│   ├── te  : HierarchicalTargetEncoder   target encoding berjenjang + kombinasi (out-of-fold)
│   ├── nb  : IdNeighborEncoder           tetangga nomor survei (out-of-fold)
│   ├── hh  : HouseholdEncoder            (opsional) bangunan serumah (out-of-fold)
│   └── adj : AdjacentAreaEncoder         (opsional) ward / blok tetangga
└── clf    : LGBMClassifier
```

### Perubahan iterasi 3 (berdasarkan EDA)
| Langkah | Bukti dari EDA |
|---|---|
| `neighbor_v2`: jarak dihitung per rumah tangga (ID // 10), bangunan serumah dikeluarkan, k = 1, 2, 5, 10, 30 | serumah *kurang* mirip (0,40) dari acak (0,45); rumah tangga sebelah paling mirip (0,53) |
| `household`: grade bangunan lain serumah | 8,6% bangunan punya saudara serumah; bangunan tambahan −0,22 grade |
| `adjacent_area`: kerusakan ward/blok dengan kode sebelah | selisih id6 berurutan 0,335 vs acak 0,385 |
| `extra_combos`: wilayah × lantai atas, wilayah × umur, id9 × material | efek setelah lokasi dikontrol: 0,24 / 0,24 / material terkuat |
| `kmeans_btype` (eksperimen) | memadatkan 6 fitur konstruksi menjadi tipe bangunan |
| `vuln_vtype` (eksperimen) | tipe bangunan dari skor kerentanan supervised |

**Ablation:** setiap langkah diuji berurutan pada subsampel 30% wilayah (3-fold) dan hanya dipertahankan bila
macro F1 *held-out* naik (clustering harus naik ≥ 0,002).

**Estimasi waktu (CPU Kaggle):** ablation ±30–40 menit, CV 5-fold ±40–50 menit, model final ±10 menit.

In [ ]:
# === Setup: temukan folder repo (lokal / Kaggle) ===
import sys, subprocess
from pathlib import Path

# Jika dijalankan di Kaggle dan repo belum ada, isi URL GitHub repo kamu (Internet harus ON):
REPO_URL = "https://github.com/yafifaldin/structural-damage-classification.git"   # contoh: "https://github.com/username/structural-damage-classification.git"

def find_repo_root():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/repo")]
    kin = Path("/kaggle/input")
    if kin.exists():
        cands += [p.parent.parent for p in kin.glob("**/scripts/__init__.py")]
    for c in cands:
        if (c / "scripts" / "__init__.py").exists():
            return c.resolve()
    return None

ROOT = find_repo_root()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    ROOT = Path("/kaggle/working/repo")
assert ROOT is not None, "Folder repo (berisi scripts/) tidak ditemukan. Isi REPO_URL atau upload repo sebagai Kaggle Dataset."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import json, time
import numpy as np
import pandas as pd
from IPython.display import display, Image

from scripts.preprocess import load_data, split_xy, output_dir, resolve_config, DEFAULT_CONFIG
from scripts import train as T

# ---- konfigurasi ----
ABLATION = True        # pilih konfigurasi fitur otomatis
ABLATION_FRAC = 0.30   # fraksi wilayah untuk ablation
TUNE = False           # RandomizedSearchCV (tambahan ±25 menit)
N_ITER = 8
TUNE_FRAC = 0.25
N_SPLITS = 5
CONFIG = dict(DEFAULT_CONFIG)   # dipakai bila ABLATION = False

OUT = output_dir()
train, test, sample = load_data()
X, y = split_xy(train)
print(X.shape, np.bincount(y)[1:])

## 1. Ablation fitur

In [ ]:
if ABLATION:
    t0 = time.time()
    CONFIG, ablation = T.run_ablation(X, y, frac=ABLATION_FRAC)
    ablation.to_csv(OUT / "reports" / "ablation_results.csv", index=False)
    print(f"ablation selesai dalam {(time.time() - t0) / 60:.1f} menit")
    display(ablation)
CONFIG = resolve_config(CONFIG)
print("konfigurasi terpilih:", CONFIG)

## 2. Pipeline terpilih

In [ ]:
pipe = T.build_pipeline(config=CONFIG)
pipe

## 3. Hyperparameter tuning (opsional)

In [ ]:
best_params = {}
if TUNE:
    t0 = time.time()
    best_params, tuning_results = T.tune_hyperparameters(X, y, config=CONFIG, n_iter=N_ITER, frac=TUNE_FRAC)
    tuning_results.to_csv(OUT / "reports" / "tuning_results.csv", index=False)
    print(f"tuning selesai dalam {(time.time() - t0) / 60:.1f} menit")
    display(tuning_results.head(10))
print("parameter LightGBM (override):", best_params or "default")

## 4. Cross-validation 5-fold pada data penuh

In [ ]:
t0 = time.time()
oof, fold_ids, classes = T.cross_validate_oof(X, y, best_params, CONFIG, n_splits=N_SPLITS)
T.save_oof(oof, fold_ids, y)
print(f"CV selesai dalam {(time.time() - t0) / 60:.1f} menit")

## 5. Optimasi bobot kelas untuk macro F1

In [ ]:
cv = T.summarize_cv(oof, y, fold_ids, classes)
print("macro F1 argmax           :", cv["macro_f1_argmax"])
print("macro F1 + bobot (OOF)    :", cv["macro_f1_weighted_oof"])
print("macro F1 held-out per fold:", cv["macro_f1_heldout_per_fold"])
print("macro F1 held-out (rata2) :", cv["macro_f1_heldout_mean"])
print("bobot kelas               :", cv["class_weights"])
display(pd.DataFrame(cv["report"]).T.round(4))
pred_oof = classes[(oof * np.asarray(cv["class_weights"])).argmax(1)]
T.plot_confusion(y, pred_oof)
Image(str(OUT / "reports" / "figures" / "confusion_matrix.png"))

## 6. Model final + simpan

In [ ]:
t0 = time.time()
model = T.fit_final_model(X, y, best_params, cv["class_weights"], CONFIG)
model_path = T.save_model(model)
print(f"model final: {(time.time() - t0) / 60:.1f} menit -> {model_path}")
imp = T.plot_feature_importance(model)
Image(str(OUT / "reports" / "figures" / "feature_importance.png"))

In [ ]:
summary = {"config": CONFIG, "params": {**T.DEFAULT_LGBM_PARAMS, **best_params},
           **{k: v for k, v in cv.items() if k != "report"},
           "per_class_f1": {k: round(v["f1-score"], 4) for k, v in cv["report"].items()
                            if k not in ("accuracy", "macro avg", "weighted avg")},
           "top_features": imp.head(20).round(0).to_dict()}
with open(OUT / "reports" / "cv_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("===== RINGKASAN ITERASI 3 (salin semua di bawah ini) =====")
if ABLATION:
    print(ablation[["step", "macro_f1_heldout", "gain", "kept"]].to_string(index=False))
print(json.dumps({k: summary[k] for k in ("config", "macro_f1_argmax", "macro_f1_heldout_mean",
                                          "per_class_f1", "class_weights", "top_features")}, default=str))

Lanjut ke `03_submission.ipynb` (atau jalankan sel di bawah untuk langsung membuat submission di sesi yang sama).

In [ ]:
from scripts.predict import predict_grades, format_submission, save_submission
sub = format_submission(sample, test["building_id"], predict_grades(model, test))
print("submission:", save_submission(sub), sub.shape)
sub["damage_grade"].value_counts(normalize=True).sort_index().round(3)